# NEU-ESC — smoke test đo thời gian trên Kaggle T4

Notebook này chạy **cả 8 setup** (2 backbone × ST sentiment, ST topic, 2-task, 2-task + SMART), mỗi setup **1 epoch rút gọn 30 step**, rồi:
- đo thời gian mỗi step, thời gian eval val/test và VRAM đỉnh;
- ngoại suy ra thời gian một run đầy đủ và tổng GPU-giờ cho 40 run (5 seed).

Kết quả metrics trong smoke **không có ý nghĩa** (model mới train 30 step) và được đánh dấu `benchmark_eligible=false`.

**Cài đặt notebook trước khi chạy:** Accelerator = **GPU T4 x2**, Internet = **On** (tài khoản cần xác minh số điện thoại). Dự kiến khoảng 20–40 phút.

In [ ]:
import glob, os, shutil, subprocess

WORK = "/kaggle/working"
CODE_DIR = f"{WORK}/neu-esc"
DATA_DIR = f"{WORK}/data"
# Cách 1: upload thư mục kaggle/ của repo thành một Kaggle Dataset rồi "Add Input" vào notebook.
# Cách 2: clone từ GitHub (repo phải public, hoặc dùng token qua Kaggle Secrets).
REPO_URL = "https://github.com/AI-Peak/neu-esc-reproduction.git"
BRANCH = "main"

uploaded = glob.glob("/kaggle/input/**/neu_esc_kaggle.py", recursive=True)
os.makedirs(f"{CODE_DIR}/kaggle", exist_ok=True)
if uploaded:
    for path in glob.glob(os.path.join(os.path.dirname(uploaded[0]), "*.py")):
        shutil.copy(path, f"{CODE_DIR}/kaggle/")
    print("code from dataset:", os.path.dirname(uploaded[0]))
else:
    shutil.rmtree(CODE_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE_DIR], check=True)
    print("code from git:", subprocess.check_output(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], text=True).strip())
RUNNER = f"{CODE_DIR}/kaggle/neu_esc_kaggle.py"
QUEUE = f"{CODE_DIR}/kaggle/run_queue.py"
AGGREGATE = f"{CODE_DIR}/kaggle/aggregate.py"
CHECK = f"{CODE_DIR}/kaggle/check_runs.py"
print(sorted(os.listdir(f"{CODE_DIR}/kaggle")))

In [ ]:
# Giữ transformers 4.x (pipeline đã kiểm với 4.x); sentencepiece cần cho tokenizer ViSoBERT.
!pip install -q "transformers>=4.44,<5" sentencepiece
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch, transformers, sklearn; print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__, '| GPUs', torch.cuda.device_count())"

In [ ]:
# Tải 6 CSV của tác giả (commit b5637cc) và kiểm SHA-256.
!python {RUNNER} fetch --data-dir {DATA_DIR}

In [ ]:
# 8 smoke run, chia đều cho 2 GPU. Log từng run nằm trong smoke/logs/.
SMOKE_DIR = f"{WORK}/smoke"
SMOKE_STEPS = 30
!python {QUEUE} --data-dir {DATA_DIR} --out-dir {SMOKE_DIR} --seeds 42 -- --epochs 1 --max-steps-per-epoch {SMOKE_STEPS} --force

In [ ]:
import json
from pathlib import Path
import pandas as pd

EPOCHS = {"st_sentiment": 15, "st_topic": 15, "mtl": 20, "mtl_smart": 20}
SEEDS = 5
rows = []
for path in sorted(Path(SMOKE_DIR).glob("*/metrics.json")):
    metrics = json.loads(path.read_text())
    config = json.loads((path.parent / "config.json").read_text())
    epoch = json.loads((path.parent / "epochs.jsonl").read_text().splitlines()[0])
    timing = metrics["timing"]
    # Phần còn lại của wall time: tokenize, load model, khởi tạo CUDA.
    overhead = timing["wall_seconds"] - epoch["train_seconds"] - epoch["val_seconds"] - timing["test_seconds"]
    epochs = EPOCHS[config["setup"]]
    full_epoch = timing["projected_train_epoch_seconds"]
    run_hours = (epochs * (full_epoch + epoch["val_seconds"]) + timing["test_seconds"] + overhead) / 3600
    rows.append({"run": metrics["run_name"], "micro_batch": config["micro_batch"],
                 "step_s": round(timing["step_seconds_mean"], 2),
                 "epoch_min": round(full_epoch / 60, 1), "val_s": round(epoch["val_seconds"]),
                 "test_s": round(timing["test_seconds"]), "overhead_s": round(overhead),
                 "peak_vram_gb": round(epoch["peak_vram_gb"] or 0, 1),
                 "epochs": epochs, "run_hours": round(run_hours, 2)})
table = pd.DataFrame(rows)
display(table)
table.to_csv(f"{SMOKE_DIR}/timing_projection.csv", index=False)


def makespan(hours, gpus=2):
    """Longest-first greedy split, the same order run_queue.py uses."""
    loads = [0.0] * gpus
    for h in sorted(hours, reverse=True):
        loads[loads.index(min(loads))] += h
    return max(loads)


one_seed = list(table["run_hours"])
print(f"1 seed x 8 setup : {sum(one_seed):.1f} GPU-giờ, ~{makespan(one_seed):.1f} giờ session với T4 x2")
print(f"{SEEDS} seed (40 run): {SEEDS * sum(one_seed):.1f} GPU-giờ, ~{makespan(one_seed * SEEDS):.1f} giờ session với T4 x2")
print("ST tính đủ 15 epoch; early stopping (patience 4) thường dừng sớm hơn nên đây là cận trên.")

In [ ]:
# Gợi ý micro-batch: VRAM đỉnh còn dư nhiều thì tăng micro-batch (ít bước tích lũy hơn, chạy nhanh hơn).
for row in rows:
    advice = "có thể thử micro-batch x2" if row["peak_vram_gb"] < 6.5 and row["micro_batch"] < 128 else "giữ nguyên"
    print(f"{row['run']:<28} micro_batch {row['micro_batch']:>3}  VRAM {row['peak_vram_gb']:>5} GB  -> {advice}")

# Tùy chọn: thử SMART với micro-batch 64 (đổi PROBE = True). Nếu OOM thì giữ 32.
PROBE = False
if PROBE:
    subprocess.run(["python", RUNNER, "train", "--backbone", "phobert", "--setup", "mtl_smart", "--seed", "42",
                    "--data-dir", DATA_DIR, "--out-dir", f"{WORK}/probe", "--epochs", "1",
                    "--max-steps-per-epoch", str(SMOKE_STEPS), "--micro-batch", "64", "--force"])
    print(json.loads(Path(f"{WORK}/probe/phobert-mtl_smart-s42/metrics.json").read_text())["timing"])

## Bước tiếp theo

1. Gửi bảng `timing_projection.csv` cho nhóm, chốt micro-batch.
2. Chạy **seed 42 cho cả 8 setup** bằng notebook `run_seeds.ipynb` (đặt `SEEDS = [42]`). Notebook tự chạy `check_runs.py`; chỉ khi **cả 8 run PASS** mới chạy tiếp seed 43–46.
3. Mỗi seed còn lại chạy trong một session riêng (có thể chia cho 4 tài khoản).

**Điều kiện đi tiếp là điều kiện kỹ thuật, không phải điểm số:** run chạy xong, không lỗi, loss giảm, val macro-F1 cao hơn mức đoán toàn lớp đa số, đủ dự đoán theo ID, mọi run cùng một phiên bản code.
Không duyệt theo việc điểm có gần Table 6 hay không. Nếu lệch xa paper thì chỉ đi tìm lỗi, không chỉnh cấu hình để đuổi theo số của paper. Nếu buộc phải đổi protocol thì tạo cohort mới và ghi quyết định trước khi xem điểm test.